# Clean IVV Wine Production Data

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FernandaChacara/pml_final_project/blob/main/notebooks/ivv_wine_production.ipynb)

This notebook cleans the IVV wine production file and saves a processed CSV with one row per wine region and campaign year.

## 1. Install Packages

In [ ]:
!pip install -q pandas numpy openpyxl xlrd lxml scikit-learn

## 2. Clone or Update Repository

In [ ]:
from pathlib import Path
import re
import shutil
import subprocess
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

REPO_URL = "https://github.com/FernandaChacara/pml_final_project.git"
ROOT_DIR = Path("/content/pml_final_project")

if ROOT_DIR.exists():
    print("Repository already exists. Pulling latest changes...")
    %cd /content/pml_final_project
    !git pull
else:
    !git clone {REPO_URL} /content/pml_final_project
    %cd /content/pml_final_project

RAW_XLS = ROOT_DIR / "data" / "raw" / "36_Evolu__o_da_Produ__o_Nacional_por_Reg7.xls"
RAW_XLSX = ROOT_DIR / "data" / "raw" / "36_Evolu__o_da_Produ__o_Nacional_por_Reg7.xlsx"
OUTPUT_FILE = ROOT_DIR / "data" / "processed" / "wine_production_by_region_clean.csv"

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

print("Root directory:", ROOT_DIR)
print("Raw XLS file:", RAW_XLS)
print("Raw XLSX file:", RAW_XLSX)
print("Output file:", OUTPUT_FILE)

## 3. Prepare the Excel File

The original `.xls` can fail in Colab with a workbook encryption error. This notebook prefers the converted `.xlsx` file. If it is missing, it tries to convert the `.xls` file with LibreOffice.

In [ ]:
def ensure_xlsx_file(raw_xls: Path, raw_xlsx: Path) -> Path:
    if raw_xlsx.exists():
        print(f"Using existing XLSX file: {raw_xlsx}")
        return raw_xlsx

    if not raw_xls.exists():
        raise FileNotFoundError(
            f"Neither {raw_xlsx.name} nor {raw_xls.name} was found in data/raw. "
            "Upload the raw IVV file to data/raw and run this notebook again."
        )

    print("Converted XLSX file was not found. Trying to convert the XLS file with LibreOffice...")

    if shutil.which("libreoffice") is None:
        print("LibreOffice is not installed. Installing it in Colab...")
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(["apt-get", "install", "-y", "-qq", "libreoffice"], check=True)

    result = subprocess.run(
        [
            "libreoffice",
            "--headless",
            "--convert-to",
            "xlsx",
            "--outdir",
            str(raw_xls.parent),
            str(raw_xls),
        ],
        capture_output=True,
        text=True,
    )

    if result.returncode != 0 or not raw_xlsx.exists():
        raise RuntimeError(
            "Could not convert the original .xls file to .xlsx in Colab.\n\n"
            "Please manually open the .xls file in Excel, save it as .xlsx, "
            "and upload it to data/raw with this exact name:\n"
            f"{raw_xlsx.name}\n\n"
            f"LibreOffice stdout:\n{result.stdout}\n\n"
            f"LibreOffice stderr:\n{result.stderr}"
        )

    print(f"Created XLSX file: {raw_xlsx}")
    return raw_xlsx


EXCEL_FILE = ensure_xlsx_file(RAW_XLS, RAW_XLSX)

## 4. Read Raw Workbook

In [ ]:
raw_sheets = pd.read_excel(
    EXCEL_FILE,
    sheet_name=None,
    header=None,
    dtype=str,
    engine="openpyxl",
)

print(f"Loaded {len(raw_sheets)} sheet(s).")
for sheet_name, sheet_df in raw_sheets.items():
    print(f"{sheet_name}: {sheet_df.shape}")

## 5. Define Cleaning Helpers

In [ ]:
CAMPAIGN_PATTERN = re.compile(r"^20\d{2}/\d{2}$")

REGIONS_TO_KEEP = [
    "Verdes",
    "T. Montes",
    "Douro",
    "Bairrada",
    "Dão",
    "Beira Interior",
    "Távora-Varosa",
    "Tejo",
    "Lisboa",
    "P. Setúbal",
    "Alentejo",
    "Algarve",
    "Madeira",
    "Açores",
]

TYPE_TO_COLUMN = {
    "total": "total_production_hl",
    "dop": "dop_production_hl",
    "igp": "igp_production_hl",
    "year_variety": "year_variety_production_hl",
    "non_certified": "non_certified_production_hl",
}


def normalize_text(value):
    if pd.isna(value):
        return ""
    text = str(value).replace("\xa0", " ").strip()
    text = re.sub(r"\s+", " ", text)
    return text


def normalize_for_matching(value):
    text = normalize_text(value).lower()
    replacements = {
        "ã": "a",
        "á": "a",
        "à": "a",
        "â": "a",
        "ç": "c",
        "é": "e",
        "ê": "e",
        "í": "i",
        "ó": "o",
        "ô": "o",
        "õ": "o",
        "ú": "u",
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    return text


REGION_MATCH = {normalize_for_matching(region): region for region in REGIONS_TO_KEEP}


def clean_number_pt(value):
    text = normalize_text(value)
    if text == "" or text.lower() in {"nan", "none"}:
        return np.nan

    text = re.sub(r"[^0-9,.-]", "", text)
    if text in {"", "-", ".", ","}:
        return np.nan

    text = text.replace(".", "").replace(",", ".")
    try:
        return float(text)
    except ValueError:
        return np.nan


def infer_table_type(title_text):
    title = normalize_for_matching(title_text)

    if "producao total por regiao" in title:
        return "total"
    if "denominacao de origem protegida" in title or " dop " in f" {title} ":
        return "dop"
    if "indicacao geografica protegida" in title or " igp " in f" {title} ":
        return "igp"
    if "ano/casta" in title or "ano / casta" in title:
        return "year_variety"
    if "sem do/ig" in title or "sem do / ig" in title:
        return "non_certified"

    return None


def detect_header_rows(df):
    header_rows = []
    for row_idx in range(len(df)):
        values = [normalize_text(value) for value in df.iloc[row_idx].tolist()]
        campaign_count = sum(bool(CAMPAIGN_PATTERN.match(value)) for value in values)
        if campaign_count >= 3:
            header_rows.append(row_idx)
    return header_rows


def find_title_above(df, header_row, max_rows_above=10):
    start = max(0, header_row - max_rows_above)
    titles = []

    for row_idx in range(start, header_row):
        row_text = " ".join(normalize_text(value) for value in df.iloc[row_idx].tolist())
        row_key = normalize_for_matching(row_text)
        if "evolucao" in row_key or "producao" in row_key:
            titles.append(row_text)

    return titles[-1] if titles else ""


def find_region_column(df, header_row):
    best_col = None
    best_count = 0
    scan_end = min(len(df), header_row + 80)

    for col_idx in range(df.shape[1]):
        matches = 0
        for row_idx in range(header_row + 1, scan_end):
            value_key = normalize_for_matching(df.iat[row_idx, col_idx])
            if value_key in REGION_MATCH:
                matches += 1

        if matches > best_count:
            best_col = col_idx
            best_count = matches

    if best_col is None or best_count == 0:
        raise ValueError(f"Could not find the region column below header row {header_row}.")

    return best_col


def collect_region_rows(df, header_row, region_col):
    rows = []

    for row_idx in range(header_row + 1, len(df)):
        region_key = normalize_for_matching(df.iat[row_idx, region_col])

        if region_key in {"total geral", "total"}:
            break

        if region_key in REGION_MATCH:
            rows.append((row_idx, REGION_MATCH[region_key]))

    return rows


def select_volume_year_columns(df, raw_year_columns, region_rows):
    selected_columns = {}

    for campaign_year in sorted(set(raw_year_columns.values()), reverse=True):
        candidate_cols = [col for col, year in raw_year_columns.items() if year == campaign_year]

        if len(candidate_cols) == 1:
            selected_columns[candidate_cols[0]] = campaign_year
            continue

        scores = {}
        for col_idx in candidate_cols:
            values = [
                clean_number_pt(df.iat[row_idx, col_idx])
                for row_idx, _region in region_rows
            ]
            values = [value for value in values if pd.notna(value)]

            if values:
                scores[col_idx] = max(abs(value) for value in values)
            else:
                scores[col_idx] = -1

        volume_col = max(scores, key=scores.get)
        selected_columns[volume_col] = campaign_year

    return dict(sorted(selected_columns.items()))

## 6. Extract IVV Production Tables

In [ ]:
def parse_table_from_header(df, sheet_name, header_row):
    title = find_title_above(df, header_row)
    table_type = infer_table_type(title)

    if table_type is None:
        print(f"Skipping sheet {sheet_name}, row {header_row}: could not identify table title.")
        print("Title found:", title[:160])
        return None

    header_values = [normalize_text(value) for value in df.iloc[header_row].tolist()]
    raw_year_columns = {
        col_idx: value
        for col_idx, value in enumerate(header_values)
        if CAMPAIGN_PATTERN.match(value)
    }

    if not raw_year_columns:
        return None

    region_col = find_region_column(df, header_row)
    region_rows = collect_region_rows(df, header_row, region_col)

    if not region_rows:
        print(f"No region rows parsed for sheet {sheet_name}, row {header_row}.")
        return None

    year_columns = select_volume_year_columns(df, raw_year_columns, region_rows)
    output_column = TYPE_TO_COLUMN[table_type]
    records = []

    for row_idx, region in region_rows:
        for col_idx, campaign_year in year_columns.items():
            records.append(
                {
                    "source_sheet": sheet_name,
                    "production_type": table_type,
                    "production_column": output_column,
                    "region": region,
                    "campaign_year": campaign_year,
                    "year_start": int(campaign_year[:4]),
                    "production_hl": clean_number_pt(df.iat[row_idx, col_idx]),
                }
            )

    if not records:
        print(f"No region rows parsed for sheet {sheet_name}, row {header_row}.")
        return None

    parsed = pd.DataFrame(records)
    print(f"Parsed {table_type}: {parsed.shape[0]} rows from sheet {sheet_name}, row {header_row}")
    return parsed


long_tables = []

for sheet_name, sheet_df in raw_sheets.items():
    header_rows = detect_header_rows(sheet_df)
    print(f"{sheet_name}: detected header rows {header_rows}")

    for header_row in header_rows:
        parsed_table = parse_table_from_header(sheet_df, sheet_name, header_row)
        if parsed_table is not None:
            long_tables.append(parsed_table)

if not long_tables:
    raise ValueError(
        "No IVV production tables were parsed. Check whether the workbook contains "
        "campaign-year columns such as 2025/26 and the expected IVV table titles."
    )

production_long = pd.concat(long_tables, ignore_index=True)

display(production_long.head())
print("Long dataset shape:", production_long.shape)

display(
    production_long.groupby("production_type")
    .agg(
        rows=("production_hl", "size"),
        regions=("region", "nunique"),
        campaign_years=("campaign_year", "nunique"),
        missing_values=("production_hl", lambda values: values.isna().sum()),
    )
    .reset_index()
)

## 7. Build Final Processed Dataset

In [ ]:
value_columns = list(TYPE_TO_COLUMN.values())

production_clean = (
    production_long.pivot_table(
        index=["region", "campaign_year", "year_start"],
        columns="production_column",
        values="production_hl",
        aggfunc="first",
    )
    .reset_index()
)

production_clean.columns.name = None

for column in value_columns:
    if column not in production_clean.columns:
        production_clean[column] = np.nan

production_clean = production_clean[["region", "campaign_year", "year_start"] + value_columns]
production_clean = production_clean.sort_values(["year_start", "region"]).reset_index(drop=True)

for column in value_columns:
    production_clean[column] = production_clean[column].round().astype("Int64")

display(production_clean.head(20))
print("Final dataset shape:", production_clean.shape)

## 8. Quality Checks

In [ ]:
print("Dataset shape:", production_clean.shape)

print("\nColumns:")
print(production_clean.columns.tolist())

print("\nUnique campaign years:")
print(sorted(production_clean["campaign_year"].unique()))

print("\nUnique regions:")
print(sorted(production_clean["region"].unique()))

print("\nDuplicated region-year rows:")
print(production_clean.duplicated(subset=["region", "campaign_year"]).sum())

print("\nMissing values:")
display(production_clean.isna().sum().to_frame("missing_count"))

print("\nSummary statistics for total_production_hl:")
display(production_clean["total_production_hl"].describe())

## 9. Save Processed CSV

In [ ]:
production_clean.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")

print("Saved processed dataset to:")
print(OUTPUT_FILE)

## 10. Quick Modelling Check

This is only a quick modelling check to confirm the processed dataset can be used in a scikit-learn workflow. It is not final modelling.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

model_data = production_clean.dropna(subset=["total_production_hl"]).copy()

feature_columns = [
    "region",
    "year_start",
    "dop_production_hl",
    "igp_production_hl",
    "year_variety_production_hl",
    "non_certified_production_hl",
]
target_column = "total_production_hl"

recent_years = sorted(model_data["year_start"].unique())[-3:]
train_data = model_data[~model_data["year_start"].isin(recent_years)]
test_data = model_data[model_data["year_start"].isin(recent_years)]

if train_data.empty or test_data.empty:
    raise ValueError("Not enough years are available for a chronological train/test split.")

numeric_features = [
    "year_start",
    "dop_production_hl",
    "igp_production_hl",
    "year_variety_production_hl",
    "non_certified_production_hl",
]
categorical_features = ["region"]

preprocess = ColumnTransformer(
    transformers=[
        ("region", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("numeric", SimpleImputer(strategy="median"), numeric_features),
    ]
)

model = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("model", LinearRegression()),
    ]
)

X_train = train_data[feature_columns]
y_train = train_data[target_column]
X_test = test_data[feature_columns]
y_test = test_data[target_column]

model.fit(X_train, y_train)
predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print("Test years:", recent_years)
print(f"MAE: {mae:,.2f}")
print(f"RMSE: {rmse:,.2f}")
print(f"R²: {r2:,.4f}")

## 11. Download CSV

In [ ]:
from google.colab import files

files.download(str(OUTPUT_FILE))